# Pulse IQB Research: Interim Progress Review

**Purpose:** A single, mentor-facing review of the work completed so far, the reasoning behind the sample-size analysis, the main findings, and the next research steps.

This notebook is designed to be shared in Google Colab. It is a narrative review, not a replacement for the detailed analysis notebooks.

## 1. Research question and proposal context

The broader Pulse project asks whether the Internet Quality Barometer (IQB) is a stable and policy-relevant measure of Internet connectivity in countries with heterogeneous infrastructure. The proposal identifies four connected questions:

1. Does IQB sensitivity vary by development context?
2. Where are the policy-relevant percentile and threshold tipping points?
3. Do use-case profiles need contextual adaptation?
4. Does the urban-rural gap explain score instability, and do IQB scores agree with external indicators?

The work completed so far concentrates on Phase 1: robustness, thresholds, and minimum measurement requirements. The later use-case, urban-rural, and external-validation analyses remain planned.

## 2. Progress overview

### Completed

- Analysed 24 months of M-Lab country-level aggregates.
- Covered 245 countries and 5,720 country-month observations.
- Verified the IQB configuration as ten distinct weighted threshold tests.
- Derived per-country sample-size requirements for an explicit reliability criterion.
- Examined observed month-to-month volatility.
- Evaluated median absolute deviation (MAD) as a possible reliability measure.
- Documented assumptions, limitations, and reproducibility outputs.

### Not yet completed

- Context-specific use-case weighting profiles.
- City-level and urban-rural comparisons.
- World Bank, ITU, and ISP market-concentration validation.
- Cluster-adjusted current-month confidence requirements.

## 3. Data used so far

The current analysis uses monthly M-Lab Network Diagnostic Tool aggregates from January 2024 through December 2025. Each country-month contains sample counts and nine percentile knots: p1, p5, p10, p25, p50, p75, p90, p95, and p99.

Download and upload measurements are represented separately. Latency and packet loss use the download-side sample count, while upload-throughput tests use upload-side measurements. The primary requirements are therefore reported in download-side measurements while retaining the country-specific upload/download ratio.

The main data limitation is that raw measurement-level records are not available in the current analysis. Pass rates at IQB thresholds must be reconstructed from the nine percentile knots.

In [ ]:
# Core study facts used throughout this review.
study = {
    'months': 24,
    'countries': 245,
    'country_months': 5720,
    'tolerance': 0.05,
    'confidence': 0.95,
    'headline_level': 'p50',
}
study

## 4. What IQB is measuring

IQB is a weighted average of use-case and network-requirement scores. Each requirement is binary: a metric either passes or fails a fixed threshold. Repeated requirements across use cases ask the same underlying question, so the full configuration reduces to ten distinct threshold tests.

\[\mathrm{IQB} = \sum_{k=1}^{10} w_k I_k, \qquad \sum_k w_k = 1.\]

This is the key conceptual step. IQB is not a smooth average where every small change produces a small score change. It is a step-valued classification score. A country can be measured very precisely and still be unstable if its distribution lies close to a hard threshold.

## 5. Reliability criterion

The primary planning criterion is:

\[P(|\widehat{IQB} - IQB_{target}| \leq 0.05) \geq 0.95.\]

The target is the country's 24-month pooled distribution at p50. In plain language: the estimated score should be within 0.05 IQB units of that pooled target in at least 95% of repeated samples.

This is a model-based planning requirement. It is not a guarantee for a single current month, because measurements are clustered and the underlying network population changes over time. The independent-measurement result should therefore be interpreted as a lower bound.

## 6. Methodological reasoning

For a threshold with pass probability \pi, the number of passing observations is modelled as \(X \sim \mathrm{Binomial}(n, \pi)\). The distance between the pass rate and the percentile cutoff is the margin. A small margin means that small sampling changes can flip the IQB classification.

The intuitive approximation is:

\[n \gtrsim \frac{\pi(1-\pi)z^2}{\delta^2},\qquad \delta = |\pi-(1-q)|.\]

The implemented analysis uses exact binomial and order-statistic probabilities rather than relying only on the normal approximation. It combines nested threshold tests within each metric and then sums their weighted contributions to the IQB score.

The original proposal anticipated bootstrap resampling. The exact calculation was adopted as the primary method because it avoids Monte Carlo error and fits the step-valued structure of IQB. Simulation is retained as a validation check.

## 7. Main results

For a tolerance of ±0.05 and 95% probability, evaluated at p50:

| Publication target | Median requirement | 90th-percentile requirement |
|---|---:|---:|
| p50 alone | approximately 560 | approximately 17,000 |
| p25/p50/p75 | approximately 2,400 | approximately 86,000 |
| all nine percentile levels | approximately 5,400 | approximately 114,000 |

The median requirement means that half of the countries require fewer measurements and half require more. The 90th-percentile requirement means that approximately 90% of countries require no more than that number, while the hardest 10% require more. It does not mean 90% confidence and does not refer to the p90 IQB score.

The requirement varies because countries sit at different distances from IQB thresholds. Countries near a threshold may need orders of magnitude more observations, and a country exactly on a threshold cannot be stabilized by additional observations under the hard-threshold rule.

## 8. Why MAD is not the primary reliability measure

MAD is useful for describing observed month-to-month movement because it is robust to occasional jumps. However, it is not suitable for deciding whether the sampling error is small enough.

For a discrete IQB sampling distribution, MAD can be exactly zero whenever the most likely score has probability greater than 50%. This can happen while a substantial minority of samples produce an error larger than 0.05. In one diagnostic at n=100, the median country had MAD equal to zero while approximately 29% of resamples were more than 0.05 IQB units from the target.

The analysis therefore uses the explicit tail-probability criterion for planning and reserves MAD for descriptive volatility.

## 9. Key achievements

- Reframed IQB reliability as ten linked threshold-classification problems.
- Produced per-country and per-percentile sample-size requirements.
- Quantified how percentile coverage changes the required sample size.
- Identified bias and unresolved cases caused by near-threshold countries.
- Distinguished finite-sample error from observed month-to-month movement.
- Documented the assumptions and limitations of the current evidence.
- Produced reproducible notebooks, CSV outputs, figures, and a written methodology.

## 10. Challenges and mitigation

**Pre-aggregated data.** Only percentile knots are available. This makes direct bootstrap resampling impossible and creates uncertainty when reconstructing pass rates. The mitigation is to report aggregate results cautiously and recommend direct threshold pass counts in a future pipeline.

**Dependence.** M-Lab tests can cluster by ASN, user, device, and time of day. The current sample-size results are therefore lower bounds. The next step is to estimate effective sample sizes using cluster-level information.

**Changing targets.** A pooled 24-month target is not identical to a current-month target. The current work reports these as separate questions rather than hiding temporal movement inside a confidence interval.

## 11. Remaining work

1. Add cluster-adjusted or effective-sample-size analysis.
2. Compare city-level and national-level stability to investigate urban-rural differences.
3. Add World Bank income classifications and infrastructure indicators.
4. Test context-specific use-case weighting profiles.
5. Integrate ITU broadband and ISP market-concentration indicators where coverage permits.
6. Assess rank stability, threshold crossings, and concurrent validity.
7. Confirm with mentors whether ±0.05 and 95% are the appropriate publication criteria.

## 12. Questions for mentor feedback

1. Is the 24-month pooled distribution the right target, or should the primary target be the current month?
2. Is ±0.05 IQB units with 95% probability an appropriate publication criterion?
3. Should the report prioritize p50, a small set of levels, or the full percentile curve?
4. What cluster information is available to estimate effective sample size?
5. Which use cases should receive different weights in different development contexts?
6. Which external indicators are most defensible for concurrent validation?

## 13. Reproducibility and sharing

The detailed evidence is in the repository notebooks `03-sample_size_reliability.ipynb` through `07-explainer.ipynb`, the written methodology in `METHODOLOGY.md`, and the generated outputs in `analysis_outputs/`. This Colab notebook is intended to be the concise progress review that links those materials into one narrative.

For a fully executable Colab version, upload the repository or place the generated CSV files in the Colab working directory. The narrative cells do not require the raw M-Lab data to display the current conclusions.